## List of links 

The first step of the dataset preparation is to gather a list of links to recipes from wikibooks

Let us collect 4 link sources (each contains approximately 200 links to recipes with images and ingredients)

In [1]:
source1 = "https://en.wikibooks.org/w/index.php?title=Category:Recipes_with_images&pageuntil=Dinuguan+%28Pork+Blood+Stew%29%0ADinuguan+%28Pork+Blood+Stew%29#mw-pages"
source2 = "https://en.wikibooks.org/w/index.php?title=Category:Recipes_with_images&pagefrom=Dinuguan+%28Pork+Blood+Stew%29%0ADinuguan+%28Pork+Blood+Stew%29#mw-pages"
source3 = "https://en.wikibooks.org/w/index.php?title=Category:Recipes_with_images&pagefrom=Locrio+%28Dominican+Chicken+and+Rice%29#mw-pages"
source4 = "https://en.wikibooks.org/w/index.php?title=Category:Recipes_with_images&pagefrom=Roast+Butternut%2C+Biltong+and+Brown+Rice+Salad%0ARoast+Butternut%2C+Biltong+and+Brown+Rice+Salad#mw-pages"


Firstly we will scrape links to each recipe so that we will have 4 lists, later to be merged wih one another.

In [10]:
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://en.wikibooks.org"

HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

def extract_article_links_from_category_html(url: str, session: requests.Session) -> list[str]:
    r = session.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    mw_pages = soup.select_one("#mw-pages")
    if not mw_pages:
        return []

    links = []
    for a in mw_pages.select("div.mw-category-group a[href]"):
        href = a["href"]
        if href.startswith("/wiki/") and not href.startswith("/wiki/Category:"):
            links.append(urljoin(BASE, href))
    seen = set()
    out = []
    for x in links:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


sources = [source1, source2, source3, source4]

all_by_source = {}
with requests.Session() as s:
    for url in sources:
        time.sleep(5)
        try:
            all_by_source[url] = extract_article_links_from_category_html(url, s)
            print(url, "->", len(all_by_source[url]))
        except requests.HTTPError as e:
            print("ERROR for:", url)
            print(e)
            all_by_source[url] = []


https://en.wikibooks.org/w/index.php?title=Category:Recipes_with_images&pageuntil=Dinuguan+%28Pork+Blood+Stew%29%0ADinuguan+%28Pork+Blood+Stew%29#mw-pages -> 200
https://en.wikibooks.org/w/index.php?title=Category:Recipes_with_images&pagefrom=Dinuguan+%28Pork+Blood+Stew%29%0ADinuguan+%28Pork+Blood+Stew%29#mw-pages -> 200
https://en.wikibooks.org/w/index.php?title=Category:Recipes_with_images&pagefrom=Locrio+%28Dominican+Chicken+and+Rice%29#mw-pages -> 200
https://en.wikibooks.org/w/index.php?title=Category:Recipes_with_images&pagefrom=Roast+Butternut%2C+Biltong+and+Brown+Rice+Salad%0ARoast+Butternut%2C+Biltong+and+Brown+Rice+Salad#mw-pages -> 197


In [39]:
source_links = []
for i in range(4):
    source_links_batch = list(all_by_source.values())[i]
    for link in source_links_batch:
        source_links.append(link)

## Content extraction

Now our goal is to extract from each recipe:
<ul>
<li>Title</li>
<li>Photo</li>
<li>List of ingredients</li>
</ul>

Helper function to extract a 'soup' from the html source:

In [11]:
def get_soup(url: str, session: requests.Session, sleep_s: float = 1.0) -> BeautifulSoup:
    time.sleep(sleep_s)
    r = session.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

In [12]:
def _clean_text(text):
    return text

Function to extract title:

In [13]:
def extract_title(recipe_url: str, session: requests.Session | None = None) -> str:
    owns = session is None
    s = session or requests.Session()
    try:
        soup = get_soup(recipe_url, s)
        h1 = soup.select_one("h1#firstHeading")
        if h1:
            return _clean_text(h1.get_text(" ", strip=True))
        # fallback
        title = soup.title.get_text(" ", strip=True) if soup.title else ""
        return _clean_text(title)
    finally:
        if owns:
            s.close()

In [52]:
import re, requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, unquote, quote

API="https://en.wikibooks.org/w/api.php"
BASE="https://en.wikibooks.org"
HEADERS={
    "User-Agent":"JanekOpalaRecipeScraper/1.0 (contact: your_email@example.com)","Accept-Language":"en"
    }
_CACHE={}

def _clean_text(s:str)->str:
    return re.sub(r"\s+"," ",re.sub(r"\[\s*\d+\s*\]","",s)).strip()

def _url_to_page_title(recipe_url:str)->str:
    p=urlparse(recipe_url).path
    if not p.startswith("/wiki/"): raise ValueError(recipe_url)
    return unquote(p[6:]).replace("_"," ")

def _parse(recipe_url:str, s:requests.Session)->BeautifulSoup:
    if recipe_url not in _CACHE:
        j=s.get(API,params={"action":"parse","page":_url_to_page_title(recipe_url),"prop":"text","format":"json","formatversion":"2","redirects":"1"},headers=HEADERS,timeout=30).json()
        _CACHE[recipe_url]=BeautifulSoup(j["parse"]["text"],"html.parser")
    return _CACHE[recipe_url]

def extract_title(recipe_url:str, session:requests.Session|None=None)->str:
    own=session is None; s=session or requests.Session()
    try: return _clean_text(_url_to_page_title(recipe_url))
    finally:
        if own: s.close()

def extract_photo(recipe_url:str, out_dir:str="recipe_images", session:requests.Session|None=None, width:int=1000)->str|None:
    own=session is None; s=session or requests.Session()
    try:
        soup=_parse(recipe_url,s)
        a=soup.select_one("table.infobox td.infobox-image a[href^='/wiki/File:'],td.infobox-image a[href^='/wiki/File:']")
        if not a: return None
        fn=unquote(a["href"].split("/wiki/",1)[-1]).split("File:",1)[-1].strip().replace(" ","_")
        return f"{BASE}/wiki/Special:FilePath/{quote(fn)}?width={width}"
    finally:
        if own: s.close()

def extract_ingredients(recipe_url: str, session: requests.Session | None = None) -> list[str]:
    own = session is None
    s = session or requests.Session()
    try:
        soup = _parse(recipe_url, s)
        h = soup.find(id="Ingredients")
        if h and h.name not in ("h2","h3","h4"):
            h = h.find_parent(["h2","h3","h4"])
        if not h:
            h = soup.find(lambda t: t.name in ("h2","h3","h4") and _clean_text(t.get_text()).lower().startswith("ingredients"))
        if not h:
            return []
        out = []
        for x in h.find_all_next():
            if x.name == "h2" and x is not h:
                break
            if x.name in ("ul","ol"):
                out += [_clean_text(li.get_text(" ",strip=True)) for li in x.select("li") if _clean_text(li.get_text(" ",strip=True))]
            if x.name == "table" and "wikitable" in x.get("class",[]):
                for tr in x.select("tr"):
                    tds = tr.select("td")
                    if not tds:
                        continue
                    name = _clean_text(tds[0].get_text(" ",strip=True))
                    if name and not name.isupper():
                        out.append(name)
        return out
    finally:
        if own:
            s.close()





In [28]:
import requests

test="https://en.wikibooks.org/wiki/Cookbook:1-2-3-4_Cake"
with requests.Session() as sess:
    print("title:", extract_title(test, sess))
    ingr=extract_ingredients(test, sess)
    print("ingredients:", len(ingr), ingr[:5])
    print("photo_url:", extract_photo(test, session=sess))


title: Cookbook:1-2-3-4 Cake
ingredients: 8 ['1 cup (240 g / 8.5 oz ) butter', '1 cup (240 ml / 8.1 oz) milk', '1 teaspoon vanilla extract', '2 cups (450 g / 16 oz) white granulated sugar', '3 cups (400 g / 14 oz) all-purpose flour']
photo_url: https://en.wikibooks.org/wiki/Special:FilePath/1-2-3-4_cake_slice_with_chocolate_sour_cream_icing.JPG?width=1000


In [45]:
import requests

test="https://en.wikibooks.org/wiki/Cookbook:Abula_(Nigerian_Three_Stews)"
with requests.Session() as sess:
    print("title:", extract_title(test, sess))
    ingr=extract_ingredients(test, sess)
    print("ingredients:", len(ingr), ingr[:5])
    print("photo_url:", extract_photo(test, session=sess))


title: Cookbook:Abula (Nigerian Three Stews)
ingredients: 14 ['1 cup dried beans', '2 large pieces of meat , cut into pieces', '½ cup palm oil', '1 onion', '2 teaspoons ground crayfish']
photo_url: https://en.wikibooks.org/wiki/Special:FilePath/Amala_and_Abula.jpg?width=1000


In [53]:
import requests

test="https://en.wikibooks.org/wiki/Cookbook:Afghan_Bread"
with requests.Session() as sess:
    print("title:", extract_title(test, sess))
    ingr=extract_ingredients(test, sess)
    print("ingredients:", len(ingr), ingr[:5])
    print("photo_url:", extract_photo(test, session=sess))


title: Cookbook:Afghan Bread
ingredients: 13 ['Dough', 'Water', 'Dry yeast', 'Sugar', 'All-purpose flour']
photo_url: https://en.wikibooks.org/wiki/Special:FilePath/Afghan_bread.jpg?width=1000


In [54]:
import pandas as pd
from tqdm import tqdm

recipes_df = pd.DataFrame(columns=["Title", "Ingredients", "Photo"])
for link in tqdm(source_links):
    with requests.Session() as sess:
        recipes_df.loc[len(recipes_df)] = {
            'Title': extract_title(link, sess),
            'Ingredients': extract_ingredients(link, sess),
            'Photo': extract_photo(link, sess)
        }

100%|██████████| 797/797 [08:43<00:00,  1.52it/s]


In [55]:
recipes_df

,Title,Ingredients,Photo
0,Cookbook:1-2-3-4 Cake,"[1 cup (240 g / 8.5 oz ) butter, 1 cup (240 ml...",https://en.wikibooks.org/wiki/Special:FilePath...
1,Cookbook:A Nice Cup of Tea,"[32 fl oz (1 L) hot water, 3–5 measures of loo...",https://en.wikibooks.org/wiki/Special:FilePath...
2,Cookbook:Aadun (Nigerian Corn Flour with Palm ...,"[1 cup corn flour, ¼ cup palm oil, ½ teaspoon ...",https://en.wikibooks.org/wiki/Special:FilePath...
3,User:Aaron Liu/sandbox,"[5 eggs, 80 grams (⅓ cup ) unsalted margarine,...",https://en.wikibooks.org/wiki/Special:FilePath...
4,Cookbook:Abula (Nigerian Three Stews),"[1 cup dried beans, 2 large pieces of meat , c...",https://en.wikibooks.org/wiki/Special:FilePath...
...,...,...,...
792,Cookbook:Zucchini Pie,"[1⅓ cup flour, ½ teaspoon salt, ½ cup shorteni...",https://en.wikibooks.org/wiki/Special:FilePath...
793,Cookbook:Zupa Ogórkowa (Polish Cucumber Soup),"[4 chicken wings, 1 small leek (or 1 small oni...",https://en.wikibooks.org/wiki/Special:FilePath...
794,Cookbook:Zuppa Toscana,"[1 lb spicy Italian sausage , crumbled, ½ lb s...",https://en.wikibooks.org/wiki/Special:FilePath...
795,Cookbook:Zwiebelrostbraten with Red Wine Onion...,"[1 onion , finely chopped, 2 garlic cloves, mi...",https://en.wikibooks.org/wiki/Special:FilePath...


In [57]:
recipes_df.to_csv("recipes_df.csv")